# Fase C: Consultas SQL analíticas

**Dataset:** `dengue_consolidado.parquet`

Este notebook responde seis preguntas de negocio mediante SQL ejecutado en DuckDB. El Parquet original se conserva sin modificaciones. Las vistas `fact_dengue`, `dim_tiempo`, `dim_territorio` y `dim_caso` representan, dentro del motor SQL, el hecho y sus dimensiones analíticas.

In [1]:
# Si el entorno esta recien creado, instalar previamente con:
# %pip install -r ../requirements.txt
import duckdb
import pandas as pd
from pathlib import Path

con = duckdb.connect()
rutas_candidatas = (
    Path.cwd() / 'datos' / 'procesados' / 'dengue_consolidado.parquet',
    Path.cwd().parent / 'datos' / 'procesados' / 'dengue_consolidado.parquet',
)
ruta_parquet = next((ruta.resolve() for ruta in rutas_candidatas if ruta.exists()), None)
if ruta_parquet is None:
    raise FileNotFoundError(
        'No se encontro datos/procesados/dengue_consolidado.parquet '
        f'desde {Path.cwd()}'
    )
ruta_sql = ruta_parquet.as_posix()

# Vista de hechos: una fila representa un registro notificado.
con.execute(f'''
    CREATE OR REPLACE VIEW fact_dengue AS
    SELECT
        row_number() OVER () AS id_registro,
        TRY_CAST(ANO AS INTEGER) AS ano,
        TRY_CAST(SEMANA AS INTEGER) AS semana,
        CAST(COD_DPTO_O AS VARCHAR) AS cod_dpto,
        CAST(COD_MUN_O AS VARCHAR) AS cod_municipio,
        CAST(Departamento_ocurrencia AS VARCHAR) AS departamento,
        CAST(Municipio_ocurrencia AS VARCHAR) AS municipio,
        CAST(AJUSTE AS VARCHAR) AS ajuste,
        CAST(PAC_HOS AS VARCHAR) AS pac_hos,
        CAST(SEXO AS VARCHAR) AS sexo,
        TRY_CAST(EDAD AS DOUBLE) AS edad,
        CAST(UNI_MED AS VARCHAR) AS unidad_medida,
        CAST(AREA AS VARCHAR) AS area,
        CAST(TIP_SS AS VARCHAR) AS tipo_seguridad,
        CAST(PER_ETN AS VARCHAR) AS pertenencia_etnica,
        CAST(GP_DESPLAZ AS VARCHAR) AS desplazado,
        CAST(GP_MIGRANT AS VARCHAR) AS migrante,
        CAST(GP_GESTAN AS VARCHAR) AS gestante,
        CAST(estrato AS VARCHAR) AS estrato,
        CAST(TIP_CAS AS VARCHAR) AS tipo_caso,
        CAST(nom_est_f_caso AS VARCHAR) AS establecimiento
    FROM read_parquet('{ruta_sql}')
''')

con.execute('''
    CREATE OR REPLACE VIEW dim_tiempo AS
    SELECT DISTINCT ano, semana
    FROM fact_dengue
    WHERE ano IS NOT NULL AND semana IS NOT NULL
''')

con.execute('''
    CREATE OR REPLACE VIEW dim_territorio AS
    SELECT DISTINCT cod_dpto, cod_municipio, departamento, municipio
    FROM fact_dengue
    WHERE cod_dpto IS NOT NULL OR cod_municipio IS NOT NULL
''')

con.execute('''
    CREATE OR REPLACE VIEW dim_caso AS
    SELECT DISTINCT tipo_caso, ajuste, unidad_medida
    FROM fact_dengue
''')

print(f'Parquet utilizado: {ruta_parquet}')
print(con.sql("SELECT COUNT(*) AS registros FROM fact_dengue").df().to_string(index=False))
print(con.sql('''
    SELECT COALESCE(NULLIF(TRIM(CAST(pac_hos AS VARCHAR)), ''), 'SIN DATO') AS pac_hos,
           COUNT(*) AS registros
    FROM fact_dengue
    GROUP BY 1
    ORDER BY registros DESC
''').df().to_string(index=False))

Parquet utilizado: C:\Users\Sanatiago Hernadez\Desktop\grupo-4-p1\datos\procesados\dengue_consolidado.parquet
 registros
    581071
pac_hos  registros
      2     357711
      1     223360


## Consulta 1. ¿En que combinaciones de tiempo, territorio y perfil se concentra la notificacion?

Esta consulta cumple el requisito de `GROUP BY` con mas de dos dimensiones: agrupa simultaneamente por ano, semana, departamento, area y sexo. Se muestran las 20 combinaciones con mayor numero de registros.

In [2]:
sql_1 = '''
SELECT
    ano, semana, departamento, area, sexo,
    COUNT(*) AS casos_notificados
FROM fact_dengue
GROUP BY ano, semana, departamento, area, sexo
ORDER BY casos_notificados DESC
LIMIT 20
'''
resultado_1 = con.sql(sql_1).df()
display(resultado_1)

,ano,semana,departamento,area,sexo,casos_notificados
0,2024,21,VALLE,1,F,1301
1,2024,16,VALLE,1,F,1295
2,2024,14,VALLE,1,M,1272
3,2024,15,VALLE,1,F,1272
4,2024,21,VALLE,1,M,1272
5,2024,14,VALLE,1,F,1267
6,2024,23,VALLE,1,F,1256
7,2024,22,VALLE,1,F,1244
8,2024,11,VALLE,1,F,1237
9,2024,18,VALLE,1,F,1221


**Interpretacion.** Cada fila identifica una combinacion concreta de periodo, departamento, area y sexo, por lo que permite localizar focos de notificacion con mayor detalle que un total nacional. Las primeras filas orientan la priorizacion de vigilancia semanal, pero representan registros notificados y no necesariamente incidencia poblacional, porque el Parquet no incluye poblacion denominadora.

## Consulta 2. ¿Cuales son los municipios lideres dentro de cada departamento?

Se utiliza `RANK()` como funcion de ventana para ordenar los municipios por cantidad total de registros dentro de su departamento. Los empates reciben el mismo puesto.

In [3]:
sql_2 = '''
WITH totales AS (
    SELECT
        departamento, municipio,
        COUNT(*) AS casos_notificados
    FROM fact_dengue
    GROUP BY departamento, municipio
), ranking AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY departamento
            ORDER BY casos_notificados DESC
        ) AS puesto_departamento
    FROM totales
)
SELECT *
FROM ranking
WHERE puesto_departamento <= 3
ORDER BY departamento, puesto_departamento, municipio
'''
resultado_2 = con.sql(sql_2).df()
display(resultado_2)

,departamento,municipio,casos_notificados,puesto_departamento
0,AMAZONAS,LETICIA,1654,1
1,AMAZONAS,PUERTO NARIÑO,230,2
2,AMAZONAS,LA PEDRERA (CD),66,3
3,ANTIOQUIA,MEDELLIN,6466,1
4,ANTIOQUIA,APARTADO,1713,2
...,...,...,...,...
94,VAUPES,TARAIRA,3,2
95,VAUPES,YAVARATE (CD),3,2
96,VICHADA,PUERTO CARREÑO,825,1
97,VICHADA,CUMARIBO,378,2


**Interpretacion.** El resultado entrega hasta tres municipios prioritarios por departamento y evita comparar directamente territorios de tamanos muy distintos. Estos municipios pueden servir para focalizar seguimiento, validacion de calidad del dato y acciones de respuesta, sin afirmar riesgo relativo mientras no se incorpore la poblacion de cada municipio.

## Consulta 3. ¿Que municipios concentran mas casos al unir hechos con la dimension territorial?

Esta consulta demuestra un `JOIN` explicito entre la tabla de hechos `fact_dengue` y la dimension `dim_territorio`. La dimension aporta los nombres y codigos territoriales, mientras el hecho aporta la medida agregada.

In [4]:
sql_3 = '''
SELECT
    t.cod_dpto,
    t.cod_municipio,
    t.departamento,
    t.municipio,
    COUNT(f.id_registro) AS casos_notificados,
    COUNT(DISTINCT f.semana) AS semanas_con_registros
FROM fact_dengue AS f
INNER JOIN dim_territorio AS t
    ON f.cod_dpto = t.cod_dpto
   AND f.cod_municipio = t.cod_municipio
   AND f.departamento IS NOT DISTINCT FROM t.departamento
   AND f.municipio IS NOT DISTINCT FROM t.municipio
GROUP BY t.cod_dpto, t.cod_municipio, t.departamento, t.municipio
ORDER BY casos_notificados DESC
LIMIT 20
'''
resultado_3 = con.sql(sql_3).df()
display(resultado_3)

,cod_dpto,cod_municipio,departamento,municipio,casos_notificados,semanas_con_registros
0,76,1,VALLE,CALI,51561,53
1,73,1,TOLIMA,IBAGUE,18663,53
2,8,1,ATLANTICO,BARRANQUILLA,17922,53
3,13,1,BOLIVAR,CARTAGENA,17733,53
4,50,1,META,VILLAVICENCIO,16662,53
5,68,1,SANTANDER,BUCARAMANGA,15628,53
6,54,1,NORTE SANTANDER,CUCUTA,11639,53
7,41,1,HUILA,NEIVA,10083,53
8,76,520,VALLE,PALMIRA,9272,53
9,68,276,SANTANDER,FLORIDABLANCA,8712,53


**Interpretacion.** El `JOIN` permite demostrar que el modelo separa las claves descriptivas del territorio de las observaciones de dengue. `semanas_con_registros` complementa el volumen: un municipio puede tener muchos registros concentrados en pocas semanas o una presencia mas distribuida durante el periodo observado.

## Consulta 4. ¿Como cambia semanalmente el volumen nacional de notificaciones?

Se calcula el total por ano-semana y se compara con la semana anterior mediante `LAG()`. La variacion absoluta y porcentual ayuda a detectar incrementos bruscos.

In [5]:
sql_4 = '''
WITH semanal AS (
    SELECT ano, semana, COUNT(*) AS casos_notificados
    FROM fact_dengue
    WHERE ano IS NOT NULL AND semana IS NOT NULL
    GROUP BY ano, semana
), comparacion AS (
    SELECT
        *,
        LAG(casos_notificados) OVER (ORDER BY ano, semana) AS casos_semana_anterior
    FROM semanal
)
SELECT
    ano, semana, casos_notificados, casos_semana_anterior,
    casos_notificados - casos_semana_anterior AS cambio_absoluto,
    ROUND(100.0 * (casos_notificados - casos_semana_anterior)
          / NULLIF(casos_semana_anterior, 0), 2) AS cambio_porcentual
FROM comparacion
ORDER BY ano, semana
'''
resultado_4 = con.sql(sql_4).df()
display(resultado_4)

,ano,semana,casos_notificados,casos_semana_anterior,cambio_absoluto,cambio_porcentual
0,2022,1,987,<NA>,<NA>,NaN
1,2022,2,870,987,-117,-11.85
2,2022,3,865,870,-5,-0.57
3,2022,4,776,865,-89,-10.29
4,2022,5,857,776,81,10.44
...,...,...,...,...,...,...
204,2025,49,1707,1605,102,6.36
205,2025,50,1731,1707,24,1.41
206,2025,51,1722,1731,-9,-0.52
207,2025,52,1573,1722,-149,-8.65


**Interpretacion.** Un cambio positivo indica mas registros que en la semana anterior y un cambio negativo indica una disminucion. La primera semana disponible no tiene comparacion; ademas, una variacion en notificacion puede reflejar cambios de reporte y no solo cambios en la transmision, por lo que debe interpretarse junto con contexto epidemiologico.

## Consulta 5. ¿Que perfiles de edad y sexo concentran los registros?

La edad se transforma en rangos analiticos mediante `CASE` y se cruza con sexo. Esto convierte una variable numerica en segmentos faciles de comparar para vigilancia.

In [6]:
sql_5 = '''
WITH perfiles AS (
    SELECT
        CASE
            WHEN edad IS NULL THEN 'Sin dato'
            WHEN edad < 5 THEN '0-4'
            WHEN edad < 15 THEN '5-14'
            WHEN edad < 25 THEN '15-24'
            WHEN edad < 45 THEN '25-44'
            WHEN edad < 65 THEN '45-64'
            ELSE '65 o mas'
        END AS rango_edad,
        COALESCE(NULLIF(TRIM(sexo), ''), 'Sin dato') AS sexo
    FROM fact_dengue
)
SELECT rango_edad, sexo, COUNT(*) AS casos_notificados
FROM perfiles
GROUP BY rango_edad, sexo
ORDER BY casos_notificados DESC
'''
resultado_5 = con.sql(sql_5).df()
display(resultado_5)

,rango_edad,sexo,casos_notificados
0,5-14,M,110649
1,5-14,F,98872
2,15-24,M,66628
3,25-44,F,60857
4,15-24,F,60008
5,25-44,M,57403
6,45-64,F,30575
7,45-64,M,26117
8,0-4,M,22294
9,0-4,F,21037


**Interpretacion.** La tabla permite identificar los grupos de edad y sexo con mayor carga de registros, incluyendo explícitamente los datos faltantes. El resultado describe la distribucion de notificaciones del dataset; no debe presentarse como una tasa de riesgo por grupo porque no se dispone del tamano poblacional de cada segmento.

## Consulta 6. ¿Que departamentos presentan mayor hospitalizacion registrada?

Se calcula por departamento el total de registros y la proporcion cuyo campo `PAC_HOS` indica hospitalizacion. En el Parquet solo aparecen los valores `1` y `2` (223.360 y 357.711 registros, respectivamente); para esta consulta se documenta de forma explicita la codificacion usada: `1` = hospitalizacion registrada y `2` = no hospitalizacion. Los valores distintos de `1` no se cuentan como hospitalizados.

In [7]:
sql_6 = '''
SELECT
    departamento,
    COUNT(*) AS casos_notificados,
    SUM(CASE
            WHEN UPPER(TRIM(pac_hos)) IN ('SI', 'S', '1', 'TRUE', 'T')
            THEN 1 ELSE 0
        END) AS casos_hospitalizados,
    ROUND(100.0 * SUM(CASE
            WHEN UPPER(TRIM(pac_hos)) IN ('SI', 'S', '1', 'TRUE', 'T')
            THEN 1 ELSE 0
        END) / NULLIF(COUNT(*), 0), 2) AS porcentaje_hospitalizacion
FROM fact_dengue
GROUP BY departamento
HAVING COUNT(*) >= 10
ORDER BY porcentaje_hospitalizacion DESC, casos_notificados DESC
'''
resultado_6 = con.sql(sql_6).df()
display(resultado_6)

,departamento,casos_notificados,casos_hospitalizados,porcentaje_hospitalizacion
0,CESAR,15219,12321.0,80.96
1,SUCRE,16569,10479.0,63.24
2,GUAJIRA,10538,6605.0,62.68
3,EXTERIOR,1101,642.0,58.31
4,MAGDALENA,10385,5259.0,50.64
5,CAQUETA,7121,3497.0,49.11
6,CORDOBA,22437,10923.0,48.68
7,ATLANTICO,31548,15163.0,48.06
8,NORTE SANTANDER,22987,10576.0,46.01
9,CUNDINAMARCA,20908,9524.0,45.55


**Interpretacion.** Cesar presenta la mayor proporcion de hospitalizacion registrada (80,96%), seguido por Sucre (63,24%) y Guajira (62,68%). Este indicador describe la distribucion de hospitalizacion dentro de los registros notificados y no equivale a gravedad poblacional ni a riesgo, porque no hay denominadores de poblacion ni informacion clinica adicional.

In [9]:
validaciones = {
    'Consulta 1': (resultado_1, {'ano', 'semana', 'departamento', 'area', 'sexo', 'casos_notificados'}),
    'Consulta 2': (resultado_2, {'departamento', 'municipio', 'casos_notificados', 'puesto_departamento'}),
    'Consulta 3': (resultado_3, {'cod_dpto', 'cod_municipio', 'departamento', 'municipio', 'casos_notificados', 'semanas_con_registros'}),
    'Consulta 4': (resultado_4, {'ano', 'semana', 'casos_notificados', 'casos_semana_anterior', 'cambio_absoluto', 'cambio_porcentual'}),
    'Consulta 5': (resultado_5, {'rango_edad', 'sexo', 'casos_notificados'}),
    'Consulta 6': (resultado_6, {'departamento', 'casos_notificados', 'casos_hospitalizados', 'porcentaje_hospitalizacion'}),
}

for nombre, (resultado, columnas_esperadas) in validaciones.items():
    assert not resultado.empty, f'{nombre} no devolvio filas'
    assert columnas_esperadas <= set(resultado.columns), f'{nombre} no tiene todas las columnas esperadas'

print(f'Validacion OK: {len(validaciones)} consultas con resultados y columnas esperadas.')

Validacion OK: 6 consultas con resultados y columnas esperadas.


## Cierre

Las seis consultas responden preguntas de negocio, devuelven tablas y quedan acompañadas por interpretaciones de 2-3 frases. El notebook trabaja directamente sobre `dengue_consolidado.parquet`, sin alterar el archivo de origen.